In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import requests
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,TimestampType
from pyspark.sql import Row

bronze_path = "/Volumes/sources_catalog/bronze/raw_data/china_moa_prices"

products = ["畜禽产品", "蔬菜", "水产品", "水果"]
now = datetime.now()
datestring = f"{now.year}-{now.month:02d}"

base_url = (
    "http://zdscxx.moa.gov.cn:8080/nyb/getFrequencyData"
    "?page=1&rows=10000&type=月度数据"
    "&subType=农产品批发价格"
    "&level=0&time=[\"2016-07\",\"{0}\"]&product={1}"
)

rows = []

for product in products:
    r = requests.post(base_url.format(datestring, product))
    r.raise_for_status()
    for item in r.json()["result"]["pageInfo"]["table"]:
        rows.append((
            item["time"],
            item["product"],
            item["item"],
            item["area"],
            item["value"],
            item["unit"],
            now
        ))

schema = StructType([
    StructField("time", StringType()),
    StructField("product", StringType()),
    StructField("item", StringType()),
    StructField("area", StringType()),
    StructField("value", StringType()),
    StructField("unit", StringType()),
    StructField("ingestion_ts", TimestampType())
])

bronze_df = spark.createDataFrame(rows, schema)
display(bronze_df)

bronze_df.write.format("delta") \
    .mode("append") \
    .save(bronze_path)

